# Tree Methods

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import precision_score, recall_score, mean_squared_error

<img src="https://upload.wikimedia.org/wikipedia/commons/f/ff/Decision_tree_model.png">

Tree Methods are a family of algorithms that handle both classification and regression by building a recursive tree-like structure and partitioning the samples into this structure.

## Decision Trees

A **Decision Tree** is a supervised learning algorithm employing the above. The idea is to start with a root node, containing all the data, then choose the best feature to split the data on (more on this later). We keep doing this recursively, until we've reached one of the following:
- A node is 100% pure (single class) in case of classification
- The tree reaches some depth
- Improvements to our score slow down below some threshold
- When there is a smaller number of items in a leaf than some predetermined $N$

During inference, we let the test sample "fall through" the tree into a leaf node, following the split conditions we've found when building the tree. At the end, we either output the majority class of the samples in the leaf node (for classification), or the average of the values of the samples (for regression).

### Classification

Classification trees find the best feature to split on by using **Entropy** and **Information Gain**.

**Entropy (Categorical Cross-Entropy)** $H$ measures how pure the data is - in a binary setting, a dataset containing only items belonging to a single class has entropy of $0$ ($100\%$ purity), whereas a dataset containing a 50/50 split of classes has maximum entropy $1$ (maximum impurity). In general, the higher the entropy, the more impure the data is (entropy can go above $1$ in multiclass settings). The formula for entropy is:

$$
H(y, \hat{y}) = - \sum_{i=1}^C y_i \log(\hat{y_i})
$$

where $C$ is the number of classes, $y_i$ is the true label (a one-hot encoded vector where the entry for the correct class is $1$, rest are $0$) and $\hat{y_i}$ is the predicted probability for class $i$.

We try splitting on each feature and pick the one that lowers entropy the most. First, we calculate the entropy for splitting on each feature:

$$
E = w_1 H(p_1) + ... + w_n H(p_n) = \sum_{i=1}^n w_i H(p_i)
$$

where $p_1...p_n$ refer to nodes resulting from the split and $w_i$ is the fraction of data points falling into node $p_i$. We essentially take a weighted average of the entropies of each of the possible splits (where the weight is the number of samples in that node) and pick the one that lowers the entropy of the parent node the most. This is called **Information Gain**:

$$
G = H(p_{parent}) - E
$$

#### Building a Decision Tree
- Start with all samples in the root node
- Calculate Information Gain for each feature
    - Sort data in that feature
    - Calculate averages between the unique adjacent values in the sorted array
    - For each average, test how much it would decrease the parent entropy if we split on it
- Pick the feature and value with the best Information Gain and split the data accordingly, creating 2 nodes
- Recursively do the same in each created node until we reach a stopping criterion

### Regression

Regression Trees work similarly, except for 2 differences. First, instead of outputting the majority class of the leaf node for a test sample, we output the average of the training samples in the node. Second, we calculate the best split not by reducing impurity, but by reducing variance using the MSE:

$$
\bar{y}_m = \frac{1}{n_m} \sum_{y \in Q_m} y
$$

$$
H(Q_m) = \frac{1}{n_m} \sum_{y \in Q_m} (y - \bar{y}_m)^2
$$

In order to determine the best split, we still calculate the Information Gain as follows:

$$
E = \sum_{i=1}^n w_i H(q_i)
$$

$$
G = H(q_{parent}) - E
$$

The tree building algorithm is the same.

In [31]:
# Note: scikit-learn uses a much faster and more memory efficient C implementation

class Node:
    def __init__(self, X, y):
        self.feature = None
        self.threshold = None
        self.prediction = None
        self.left = None
        self.right = None
        
        self.score = self._score(y)
        self.n_samples = X.shape[0]
        # exit early if node is already pure
        if self._stop_early():
            self._exit(y)
            return
            
        best_info_gain = 0
        best_feature = None
        best_thr = None
        best_left_X = None
        best_right_X = None
        best_left_y = None
        best_right_y = None

        for col in range(X.shape[1]):
            order = X[:, col].argsort()
            sorted_X = X[order]
            sorted_y = y[order]
            feat_avgs = np.convolve(sorted_X[:, col], [0.5, 0.5], "valid")
            # find best value to split on
            for val in feat_avgs:
                mask = X[:, col] <= val
                left_X, right_X = X[mask], X[~mask]
                left_y, right_y = y[mask], y[~mask]
                w1, w2 = left_X.shape[0] / X.shape[0], right_X.shape[0] / X.shape[0]
                split_score = w1 * self._score(left_y) + w2 * self._score(right_y)
                if self.score - split_score > best_info_gain:
                    best_info_gain = self.score - split_score
                    best_feature = col
                    best_thr = val
                    best_left_X = left_X
                    best_right_X = right_X
                    best_left_y = left_y
                    best_right_y = right_y

        # exit early if no improvement
        if best_info_gain == 0:
            self._exit(y)
            return
        
        self.feature = best_feature
        self.threshold = best_thr
        self.prediction = None
        self.left = self.__class__(best_left_X, best_left_y)
        self.right = self.__class__(best_right_X, best_right_y)

    def _score(self, y):
        raise NotImplementedError()

    def _exit(self, y):
        self.left = None
        self.right = None
        self._assign_prediction(y)

    def _assign_prediction(self, y):
        raise NotImplementedError()

    def _stop_early(self):
        raise NotImplementedError()

class NodeC(Node):
    """Classification node"""
    def _score(self, y):
        # calculate entropy
        _, counts = np.unique(y, return_counts=True)
        p = counts / y.shape[0]
        return -np.sum(p * np.log2(p))

    def _assign_prediction(self, y):
        values, counts = np.unique(y, return_counts=True)
        ind = np.argmax(counts)
        self.prediction = values[ind]

    def _stop_early(self):
        return self.score == 0

    def __repr__(self):
        return f"NodeC(entropy={self.score}, feature={self.feature}, threshold={self.threshold}, prediction={self.prediction}, left={self.left is not None}, right={self.right is not None})"

class NodeR(Node):
    """Regression node"""
    def _score(self, y):
        # calculate MSE
        y_bar = np.mean(y)
        return np.mean((y - y_bar) ** 2)

    def _assign_prediction(self, y):
        self.prediction = np.mean(y)

    def _stop_early(self):
        return self.n_samples == 1

    def __repr__(self):
        return f"NodeR(entropy={self.score}, feature={self.feature}, threshold={self.threshold}, prediction={self.prediction}, left={self.left is not None}, right={self.right is not None})"

class DT:
    node_type = None
    def __init__(self):
        self.root = None

    def fit(self, X, y):
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        self.root = self.node_type(X, y)
        return self

    def predict(self, X):
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        result = []
        for item in X:
            node = self.root
            while node is not None:
                # we're at a leaf node, make prediction
                if node.left is None and node.right is None:
                    result.append(node.prediction)
                    break
                feature = node.feature
                threshold = node.threshold
                # traverse the tree
                if item[feature] <= threshold:
                    if node.left is not None:
                        node = node.left
                else:
                    if node.right is not None:
                        node = node.right
        return np.array(result)

class DTClf(DT):
    node_type = NodeC

class DTReg(DT):
    node_type = NodeR

In [32]:
X, y = make_classification(n_samples=2000, n_features=2, n_informative=2, n_redundant=0, n_classes=2, n_clusters_per_class=1, random_state=2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

t = DTClf().fit(X_train, y_train)

t2 = DecisionTreeClassifier(criterion='entropy', random_state=1, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                            min_weight_fraction_leaf=0.0, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0,
                           ).fit(X_train, y_train)
(t.predict(X_test) == t2.predict(X_test)).sum()

np.int64(400)

In [33]:
precision_score(y_test, t.predict(X_test)), precision_score(y_test, t2.predict(X_test))

(0.9897959183673469, 0.9897959183673469)

In [34]:
recall_score(y_test, t.predict(X_test)), recall_score(y_test, t2.predict(X_test))

(0.9948717948717949, 0.9948717948717949)

In [38]:
X, y = X, y = make_regression(n_samples=1000, n_features=4, noise=20, random_state=42, n_informative=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
t = DTReg().fit(X_train, y_train)
t2 = DecisionTreeRegressor(criterion='squared_error', random_state=1, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                        min_weight_fraction_leaf=0.0, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0
                       ).fit(X_train, y_train)
t.predict(X_test)[:10]

array([-211.50394909,   70.0549032 ,   -1.94608776,    4.59838348,
       -129.98472622,    1.88151142,   18.53681695, -152.78190125,
         96.29837327,  191.82115354])

In [41]:
t2.predict(X_test)[:10]

array([-211.50394909,   70.0549032 ,  -10.05136012,    4.59838348,
       -129.98472622,    1.88151142,   18.53681695, -111.07162975,
         96.29837327,  203.71564059])

In [42]:
mean_squared_error(y_train, t.predict(X_train)), mean_squared_error(y_train, t2.predict(X_train))

(0.0, 0.0)

In [43]:
mean_squared_error(y_test, t.predict(X_test)), mean_squared_error(y_test, t2.predict(X_test))

(1846.9139481378938, 1919.5253967826145)